# Phase 3: Feature Engineering

## The framing question that shapes everything in this notebook

Before writing a single feature, we have to answer: **when, exactly, is this
forecast made, and what information is actually available at that moment?**
This sounds pedantic but it's the single most common source of a subtly
broken (leaky) forecasting model, and it's worth being explicit about.

We're building a **next-day, hourly demand forecast**: once per day, we
predict all 24 hourly demand values for the following day. Critically, this
means the forecast for, say, 11 PM tomorrow is generated at the SAME moment
as the forecast for 1 AM tomorrow -- both are produced together, right now,
using only data through the end of today.

This has a direct, concrete consequence for feature engineering: **a lag
feature like "demand 1 hour ago" is not valid for this task.** If I'm
forecasting 11 PM tomorrow, "1 hour ago" would mean 10 PM tomorrow -- which
I don't know yet, because that's also in the future relative to when I'm
making the forecast. The only lag features that are safely available for
EVERY target hour, computed at a single consistent cutoff time, are ones
that are whole-day multiples: demand exactly 24, 48, or 168 hours ago (1
day, 2 days, 1 week prior). Those are all safely in the past no matter which
hour of tomorrow we're predicting.

This is exactly the kind of detail worth raising unprompted in an interview
-- it signals you're thinking about how the model will actually be *used*,
not just fitting a model to a static table of numbers.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/interim/cleaned_dataset.csv", parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
df.head()

## 0. Verify the cleaned data coming in

Missing-value handling (short-gap interpolation + climatological fill) was
already done in `01_data_exploration.ipynb`, and the result was saved to
`data/interim/cleaned_dataset.csv` -- that's what we just loaded above.
Deliberately NOT redoing that cleaning logic here: keeping it in exactly
one place (rather than duplicated across two notebooks) means there's a
single source of truth for how the data was cleaned, and no risk of the
two copies quietly drifting out of sync if one gets edited later.

Still worth a quick sanity check before building features on top of it,
rather than blindly trusting an upstream notebook produced exactly what we
expect.

In [ ]:
assert df['temperature_c'].isna().sum() == 0, "Unexpected missing temperature -- check 01_data_exploration.ipynb was re-run"
assert df['hoep'].isna().sum() == 0, "Unexpected missing hoep -- check 01_data_exploration.ipynb was re-run"
print("Confirmed: no missing temperature or price values coming into Phase 3.")

## 1. Lag features (demand and price)

Per the framing above: only 24h-multiple lags. I'll use 1 day, 2 days, and
1 week back. The 1-week lag is particularly valuable -- it captures
"what did this exact hour look like on the same day of the week last week,"
which implicitly encodes weekly seasonality (e.g. Tuesday-vs-Saturday
demand patterns) without me having to hand-engineer that separately.

**Why lags at all, rather than just calendar features?** Calendar features
(hour, day of week) tell the model "it's a Tuesday at 6 PM," but they can't
tell it "and this has been an unusually hot week" or "and there was a
holiday-shortened week recently disrupting the normal pattern." Lags
give the model recent *actual* behavior, which calendar features alone
cannot.

In [ ]:
LAG_HOURS = [24, 48, 168]  # 1 day, 2 days, 1 week -- all safe under our forecast framing

for lag in LAG_HOURS:
    df[f'demand_lag_{lag}h'] = df['ontario_demand'].shift(lag)
    df[f'hoep_lag_{lag}h'] = df['hoep'].shift(lag)

df[['datetime', 'ontario_demand', 'demand_lag_24h', 'demand_lag_168h']].tail(10)

## 2. Rolling statistics

Rolling means/stds smooth out hour-to-hour noise and capture a "recent
trend level" -- useful because demand isn't just a function of the calendar,
it also drifts with things like ongoing weather patterns and economic
conditions that calendar features can't capture.

**Leakage discipline again:** a naive `df['ontario_demand'].rolling(24).mean()`
would include hours from LATER THAN our forecast cutoff for some rows,
because pandas' rolling window by default looks backward from the CURRENT
row -- and "current row" here might be inside the forecast target period
itself if I'm not careful about which column I roll over. The safe move is
to compute rolling statistics on the already-lagged series (e.g. roll over
`demand_lag_24h`), not on the raw same-day series, so every value in the
window is guaranteed to be at or before our forecast cutoff.

In [ ]:
# Rolling over the lagged series, NOT the raw series -- this keeps every
# value inside the window safely in the past relative to the forecast cutoff
df['demand_rolling_24h_mean'] = df['demand_lag_24h'].rolling(24).mean()
df['demand_rolling_7d_mean'] = df['demand_lag_24h'].rolling(24 * 7).mean()
df['demand_rolling_7d_std'] = df['demand_lag_24h'].rolling(24 * 7).std()

df[['datetime', 'ontario_demand', 'demand_rolling_24h_mean', 'demand_rolling_7d_mean']].tail(10)

## 3. Calendar features

Unlike lags, calendar features carry **zero leakage risk** -- the fact that
tomorrow is a Tuesday in March is known with certainty today. These are the
easiest, safest signal in the whole feature set, and they capture demand's
strongest, most reliable patterns: the daily double-peak (work-day morning
ramp + evening peak), the weekday/weekend gap, and slower seasonal
(monthly) patterns.

**Why encode hour and month cyclically (sin/cos) instead of as plain
integers?** Hour 23 and hour 0 are one hour apart in reality, but as raw
integers they're 23 units apart -- a model would have no way to know
they're actually adjacent. Sin/cos encoding maps each cyclical variable
onto a circle, so hour 23 and hour 0 end up close together in the encoded
space, matching reality. This matters more for models like the LSTM
(Phase 4) that don't have an inherent way to discover this on their own;
tree-based models like XGBoost can partially work around it by splitting
on the raw integer repeatedly, but the cyclical encoding still tends to
help and costs nothing to include.

In [ ]:
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek  # Monday=0
df['month'] = df['datetime'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Cyclical encodings
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df[['datetime', 'hour', 'hour_sin', 'hour_cos', 'is_weekend']].head()

### Holidays

Statutory holidays disrupt the normal weekday demand pattern (a Monday
holiday behaves more like a Sunday than a typical Monday), so this is worth
flagging explicitly rather than letting the model quietly misattribute
holiday dips to some other cause. The `holidays` Python package maintains
Ontario's statutory holiday calendar, so there's no need to hand-maintain
a list.

Install if needed: `uv add holidays`

In [ ]:
import holidays

on_holidays = holidays.Canada(prov='ON', years=range(2002, 2026))
df['is_holiday'] = df['datetime'].dt.date.astype(str).map(
    lambda d: pd.Timestamp(d) in on_holidays
).astype(int)

df[df['is_holiday'] == 1][['datetime', 'ontario_demand']].head()

## 4. Temperature features

Two things to add here, both standard in the energy-forecasting industry:

**Heating/Cooling Degree Hours (HDH/CDH).** Raw temperature has a
non-linear, U-shaped relationship with demand (both very cold AND very hot
weather drive demand up, via heating and cooling respectively) -- we saw
this directly in the Phase 2 scatter plot. Degree-day/degree-hour
transforms are the standard energy-industry way to linearize this: HDH
measures "how far below a comfortable baseline (typically 18°C) the
temperature is" (zero if above baseline), and CDH measures the opposite.
Feeding the model these two features, instead of raw temperature alone,
makes the U-shape explicit rather than asking the model to discover a
non-monotonic relationship on its own -- XGBoost can find non-linear
patterns regardless, but this is still a well-established, interview-worthy
domain transform to know and mention.

**An honest leakage caveat.** In a genuinely deployed system, tomorrow's
temperature is itself a FORECAST, not an observed value -- you would not
have access to the true temperature at forecast time. For this project,
using actual historical temperature as a stand-in for "the weather forecast
that would have been available" is a simplification I'm making
consciously, and it should be stated explicitly in the writeup: it means
our backtested forecast accuracy is a **best case**, assuming a perfect
weather forecast, and real deployed accuracy would be somewhat worse due to
weather forecast error compounding into the demand forecast error.

In [ ]:
BASELINE_TEMP_C = 18.0

df['heating_degree_hours'] = (BASELINE_TEMP_C - df['temperature_c']).clip(lower=0)
df['cooling_degree_hours'] = (df['temperature_c'] - BASELINE_TEMP_C).clip(lower=0)

df[['datetime', 'temperature_c', 'heating_degree_hours', 'cooling_degree_hours']].describe()

## 5. Defining the target and checking for leakage end-to-end

The target is simply `ontario_demand` at time t -- what we're trying to
predict. Before moving on, it's worth explicitly listing every feature and
confirming none of them could only be known AFTER the forecast is made.
This kind of explicit leakage audit is exactly the step that's easy to skip
under time pressure and exactly the step an interviewer will probe for if
you say "I built a forecasting model."

In [ ]:
feature_cols = [
    'demand_lag_24h', 'demand_lag_48h', 'demand_lag_168h',
    'hoep_lag_24h', 'hoep_lag_48h', 'hoep_lag_168h',
    'demand_rolling_24h_mean', 'demand_rolling_7d_mean', 'demand_rolling_7d_std',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week', 'is_weekend', 'is_holiday',
    'temperature_c', 'heating_degree_hours', 'cooling_degree_hours',
]
target_col = 'ontario_demand'

# Every one of these is either (a) a lag/rolling stat computed with a >=24h
# cutoff, (b) a calendar fact known with certainty in advance, or (c) a
# temperature value we've explicitly flagged as a "perfect forecast"
# simplification above. Nothing here peeks at same-hour-or-later actuals.
print(f"Feature count: {len(feature_cols)}")
print(feature_cols)

## 6. Dropping the warm-up period and doing a time-based train/test split

The lag/rolling features create `NaN`s for the first 7 days of the dataset
(there's no "168 hours ago" for the very first week), so those rows have to
be dropped -- losing a week out of 23 years of data is a non-issue.

**Why a time-based split, not a random shuffle:** this is worth stating
explicitly because it's a classic, easy-to-make mistake. Randomly shuffling
rows before splitting would let the model train on data from, say, next
Tuesday and get tested on last Monday -- effectively letting it see the
future during training. A real forecasting system will only ever have the
past to learn from, so the evaluation has to respect that same constraint.
I'll hold out the final 2 years (2023 through April 2025) as the test set,
and everything before that as training data.

In [ ]:
model_df = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

split_date = pd.Timestamp('2023-01-01')
train_df = model_df[model_df['datetime'] < split_date]
test_df = model_df[model_df['datetime'] >= split_date]

print(f"Train: {len(train_df):,} rows ({train_df['datetime'].min()} to {train_df['datetime'].max()})")
print(f"Test:  {len(test_df):,} rows ({test_df['datetime'].min()} to {test_df['datetime'].max()})")

## 7. Scaling -- needed for the LSTM, not for XGBoost

This is a good thing to be able to explain clearly: **tree-based models
like XGBoost are invariant to monotonic feature scaling** -- a tree just
asks "is this feature above or below some threshold," and that threshold
adapts regardless of whether the feature ranges from 0-1 or 0-20000. So
XGBoost can be trained directly on `train_df`/`test_df` as-is.

**Neural networks are different.** Gradient descent-based optimization
(what trains an LSTM) is sensitive to feature scale -- a feature ranging in
the tens of thousands (demand in MW) alongside one ranging from -1 to 1
(our sin/cos encodings) can cause uneven, unstable gradient updates. So for
the LSTM specifically, we scale features (typically to zero mean/unit
variance, or a 0-1 range).

**Leakage discipline one more time:** the scaler must be `fit` only on the
training set, then applied (`transform`, not `fit_transform`) to the test
set. Fitting on the full dataset would let information about the test set's
distribution (its min/max/mean) leak into training.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train_df[feature_cols])  # fit on TRAIN ONLY

train_scaled = scaler.transform(train_df[feature_cols])
test_scaled = scaler.transform(test_df[feature_cols])  # transform, not fit_transform

print("Scaler fit on training data only. Use train_scaled/test_scaled for the LSTM in Phase 4;")
print("use train_df[feature_cols]/test_df[feature_cols] directly for XGBoost.")

## 8. Save the processed dataset

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

model_df.to_csv('../data/processed/model_ready_dataset.csv', index=False)
train_df.to_csv('../data/processed/train.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

print(f"Saved {len(model_df):,} total rows, {len(feature_cols)} features, to data/processed/")

## Summary: what this phase actually achieved, and why it matters

Every design decision above traces back to one constraint: **the model can
only use information that would genuinely be available at the moment the
forecast is issued.** That discipline is what separates a forecasting
pipeline that will work in the real world from one that looks great in a
notebook (because it accidentally saw the future) and then fails
completely once deployed.

Next: **Phase 4**, where we train XGBoost and an LSTM on this feature set
and compare them head to head.